In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/golden_dataset.csv', parse_dates=['order_purchase_timestamp'])
rfm = pd.read_csv('../data/processed/rfm_scores.csv')
df = df.merge(rfm[['customer_unique_id', 'recency', 'monetary', 'segment']], on='customer_unique_id', how='left')

# Your churn definition — 90 days
# Churned if recency > 90 days

df['churned'] = (df['recency'] > 90).astype(int)
print('Churn rate:', df['churned'].mean().round(3) * 100, '%')

# Build features at customer level
features = df.groupby('customer_unique_id', as_index=False).agg(
    total_orders=('order_id', 'nunique'),
    total_spend=('payment_value', 'sum'),
    avg_order_value=('payment_value', 'mean'),
    avg_freight=('freight_value', 'mean'),
    avg_price=('price', 'mean'),
    avg_delivery_days=('purchased_delivered', 'mean'),
    max_installments=('payment_installments', 'max'),
    churned=('churned', 'max'),
    monetary=('monetary', 'first'),
    recency=('recency', 'first'),
    segment=('segment', 'first')
)

# Fill missing values
num_cols = features.select_dtypes(include='number').columns
features[num_cols] = features[num_cols].fillna(features[num_cols].median())

features.to_csv('../data/processed/features.csv', index=False)
print('Features saved:', features.shape)

Churn rate: 80.0 %
Features saved: (93350, 12)
